# LiDAR → Orthonormal Depth Map Generator
Generates uint8 PNG depth maps (orthonormal: z-axis depth, not Euclidean radial distance) from LiDAR point clouds projected onto the front camera plane, for **nuScenes** (.bin), **TIAND** (.pcd), and **IDD** (.npy) datasets.

**Depth encoding:** `0` = invalid (no LiDAR return), `1–255` = linearly mapped from `MIN_DEPTH_M` to `MAX_DEPTH_M`.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║                     CELL 1 — CONFIGURATION                         ║
# ╚══════════════════════════════════════════════════════════════════════╝

# ── General ──────────────────────────────────────────────────────────────
OUTPUT_ROOT         = "/STORAGE/Arka/Datasets/depth_maps_uint8"  # root output dir
MAX_DEPTH_M         = 80.0    # metres → mapped to uint8 = 255
MIN_DEPTH_M         = 0.1     # metres; points closer than this are discarded
SAVE_COLORIZED      = True    # also write jet-colourised PNGs for visual QC
VERBOSE             = True    # print per-sample progress

# ── nuScenes ─────────────────────────────────────────────────────────────
NUSCENES_ENABLED        = True
NUSCENES_DATAROOT       = "/STORAGE/Arka/Datasets/nuscenes"   # root with v1.0-*/
NUSCENES_VERSION        = "v1.0-trainval"    # "v1.0-mini" | "v1.0-trainval"
NUSCENES_CAMERA         = "CAM_FRONT"        # sensor name for front camera
NUSCENES_LIDAR          = "LIDAR_TOP"
NUSCENES_SPLIT          = "all"              # "train" | "val" | "all"
NUSCENES_MAX_SAMPLES    = None               # int to cap, None = process all
# Global-frame ego-motion compensation (recommended True for accuracy)
NUSCENES_FULL_TRANSFORM = True

# ── TIAND ────────────────────────────────────────────────────────────────
TIAND_ENABLED           = True
TIAND_ROOT              = "/STORAGE/Arka/Datasets/tiand"   # dataset root
# Expected layout per sequence:
#   <seq>/image_2/<frame>.png   — front camera images
#   <seq>/velodyne/<frame>.pcd  — LiDAR point clouds
#   <seq>/calib/calib.txt       — KITTI-style calibration  (OR use global below)
TIAND_IMG_SUBDIR        = "image_2"
TIAND_LIDAR_SUBDIR      = "velodyne"
TIAND_CALIB_SUBDIR      = "calib"            # per-sequence calib folder
TIAND_CALIB_FILENAME    = "calib.txt"        # KITTI-format calib file
# If your TIAND variant uses a single global YAML calib, set the path;
# otherwise leave None to use per-sequence KITTI calib files.
TIAND_GLOBAL_CALIB_YAML = None               # e.g. "/STORAGE/.../calib.yaml"
# Manual override — only used when both auto-load options fail:
TIAND_MANUAL_K          = [                  # 3×3 camera intrinsic
    [1266.417, 0.0,      816.267],
    [0.0,      1266.417, 491.507],
    [0.0,      0.0,      1.0    ]
]
TIAND_MANUAL_T_LID2CAM  = [                  # 4×4 lidar→camera extrinsic
    [ 0.0,  -1.0,  0.0,  0.0],
    [ 0.0,   0.0, -1.0,  0.0],
    [ 1.0,   0.0,  0.0,  0.0],
    [ 0.0,   0.0,  0.0,  1.0]
]
TIAND_MAX_SAMPLES       = None

# ── IDD ──────────────────────────────────────────────────────────────────
IDD_ENABLED             = True
IDD_ROOT                = "/STORAGE/Arka/Datasets/idd"     # dataset root
# Expected layout:
#   <root>/<split>/<sequence>/leftImg8bit/<frame>.png   — front camera
#   <root>/<split>/<sequence>/lidar/<frame>.npy         — point cloud (N,4) float32
IDD_IMG_SUBDIR          = "leftImg8bit"      # image sub-folder inside sequence
IDD_LIDAR_SUBDIR        = "lidar"            # lidar sub-folder inside sequence
IDD_SPLITS              = ["train", "val"]   # sub-dirs of IDD_ROOT to scan
# Calibration — path to a YAML or JSON file with keys:
#   camera_matrix (3×3), lidar_to_camera (4×4)
IDD_CALIB_FILE          = "/STORAGE/Arka/Datasets/idd/calib/idd_calib.yaml"
# Manual fallback (used if IDD_CALIB_FILE not found):
IDD_MANUAL_K            = [                  # 3×3 camera intrinsic
    [2252.0,    0.0,  960.0],
    [   0.0, 2252.0,  540.0],
    [   0.0,    0.0,    1.0]
]
IDD_MANUAL_T_LID2CAM    = [                  # 4×4 lidar→camera extrinsic
    [ 0.0, -1.0,  0.0,  0.0],
    [ 0.0,  0.0, -1.0, -0.08],
    [ 1.0,  0.0,  0.0, -0.27],
    [ 0.0,  0.0,  0.0,  1.0]
]
IDD_MAX_SAMPLES         = None

## Imports & Environment Check

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║                     CELL 2 — IMPORTS                               ║
# ╚══════════════════════════════════════════════════════════════════════╝

import os
import sys
import json
import glob
import struct
import warnings
import traceback
from pathlib import Path
from collections import defaultdict

import numpy as np
import cv2
import yaml
from PIL import Image
from tqdm.auto import tqdm
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# nuScenes devkit  (conda: pip install nuscenes-devkit)
if NUSCENES_ENABLED:
    try:
        from nuscenes.nuscenes import NuScenes
        from nuscenes.utils.splits import create_splits_scenes
        from pyquaternion import Quaternion
        print("✓ nuscenes-devkit + pyquaternion found")
    except ImportError as e:
        print(f"✗ nuScenes deps missing: {e}")
        NUSCENES_ENABLED = False

# Open3D for PCD reading
if TIAND_ENABLED:
    try:
        import open3d as o3d
        print(f"✓ open3d {o3d.__version__} found")
    except ImportError:
        print("✗ open3d not found — install with: pip install open3d")
        TIAND_ENABLED = False

print(f"\nDatasets enabled: nuScenes={NUSCENES_ENABLED}, TIAND={TIAND_ENABLED}, IDD={IDD_ENABLED}")
os.makedirs(OUTPUT_ROOT, exist_ok=True)

## Core Utility Functions

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║               CELL 3 — CORE PROJECTION UTILITIES                   ║
# ╚══════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────
# 3-A  Point cloud loaders
# ─────────────────────────────────────────────

def load_bin(path: str) -> np.ndarray:
    """Load KITTI-style binary float32 point cloud (N×4: x,y,z,intensity).
    Returns (N,3) xyz array."""
    pts = np.fromfile(path, dtype=np.float32).reshape(-1, 4)
    return pts[:, :3]


def load_pcd(path: str) -> np.ndarray:
    """Load PCD file (ASCII or binary) via Open3D. Returns (N,3) xyz array."""
    pcd = o3d.io.read_point_cloud(path)
    pts = np.asarray(pcd.points, dtype=np.float32)
    if pts.size == 0:
        raise ValueError(f"Empty point cloud: {path}")
    return pts  # (N,3)


def load_npy(path: str) -> np.ndarray:
    """Load numpy point cloud. Accepts shapes (N,3) or (N,4+).
    Returns (N,3) xyz array."""
    pts = np.load(path)
    if pts.ndim == 1:               # flat array → reshape
        cols = 4 if pts.size % 4 == 0 else 3
        pts = pts.reshape(-1, cols)
    return pts[:, :3].astype(np.float32)


# ─────────────────────────────────────────────
# 3-B  Orthonormal depth map projection
# ─────────────────────────────────────────────

def project_lidar_to_depth(
    points_xyz: np.ndarray,   # (N,3)  LiDAR frame
    K:          np.ndarray,   # (3,3)  camera intrinsic
    T_lid2cam:  np.ndarray,   # (4,4)  LiDAR → camera extrinsic
    img_h:      int,
    img_w:      int,
    min_depth:  float = 0.1,
    max_depth:  float = 80.0,
) -> np.ndarray:
    """
    Project LiDAR points onto the image plane and return a float32 depth map
    of shape (H, W) where each value is the *orthonormal* depth — the z-component
    of the point in camera coordinates (perpendicular distance to image plane),
    NOT the Euclidean radial distance.

    Pixels with no LiDAR return are 0.0.
    When multiple points map to the same pixel, the *closest* point wins.
    """
    N = len(points_xyz)
    if N == 0:
        return np.zeros((img_h, img_w), dtype=np.float32)

    # ── 1. Transform to camera frame ──────────────────────────────────
    pts_h = np.hstack([points_xyz, np.ones((N, 1), dtype=np.float32)])  # (N,4)
    pts_cam = (T_lid2cam.astype(np.float64) @ pts_h.T.astype(np.float64)).T  # (N,4)

    # ── 2. Orthonormal depth = z in camera frame ───────────────────────
    z = pts_cam[:, 2].astype(np.float32)

    # ── 3. Depth-range filter ──────────────────────────────────────────
    valid = (z >= min_depth) & (z <= max_depth)
    z     = z[valid]
    xyz_c = pts_cam[valid, :3].astype(np.float64)

    if z.size == 0:
        return np.zeros((img_h, img_w), dtype=np.float32)

    # ── 4. Project to image plane ──────────────────────────────────────
    uv_h = (K.astype(np.float64) @ xyz_c.T).T         # (M,3)
    u = uv_h[:, 0] / uv_h[:, 2]                        # pixel x
    v = uv_h[:, 1] / uv_h[:, 2]                        # pixel y

    # ── 5. Image boundary filter ───────────────────────────────────────
    in_bounds = (u >= 0) & (u < img_w) & (v >= 0) & (v < img_h)
    u = u[in_bounds].astype(np.int32)
    v = v[in_bounds].astype(np.int32)
    z = z[in_bounds]

    # ── 6. Scatter — closest point wins ──────────────────────────────
    # Initialise with max float; overwrite from far-to-near so nearest wins.
    depth_map = np.full((img_h, img_w), fill_value=np.inf, dtype=np.float32)
    order = np.argsort(-z)          # descending → far first, near overwrites
    depth_map[v[order], u[order]] = z[order]
    depth_map[np.isinf(depth_map)] = 0.0

    return depth_map


# ─────────────────────────────────────────────
# 3-C  uint8 encoding & saving
# ─────────────────────────────────────────────

def depth_float_to_uint8(
    depth_map: np.ndarray,
    max_depth: float = 80.0,
    min_depth: float = 0.1,
) -> np.ndarray:
    """
    Encode float32 metric depth map to uint8.

    Encoding:
        0         → invalid (no LiDAR return, or depth == 0)
        1 – 255   → linear mapping of [min_depth, max_depth] to [1, 255]

    Decode back to metres:
        depth_m = uint8_val / 255.0 * max_depth   (for valid pixels)
    """
    out = np.zeros_like(depth_map, dtype=np.uint8)
    valid = depth_map > 0.0
    # Map [min_depth, max_depth] → [1, 255]
    scaled = np.clip((depth_map[valid] / max_depth) * 255.0, 1, 255)
    out[valid] = scaled.astype(np.uint8)
    return out


def save_depth_uint8(depth_uint8: np.ndarray, out_path: str) -> None:
    """Save uint8 depth map as a single-channel PNG."""
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    img = Image.fromarray(depth_uint8, mode="L")
    img.save(out_path)


def save_depth_colorized(depth_uint8: np.ndarray, out_path: str) -> None:
    """Save a jet-colourised version of the depth map for visual QC."""
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    mask   = depth_uint8 > 0
    normed = depth_uint8.astype(np.float32) / 255.0
    color  = (cm.jet(normed)[:, :, :3] * 255).astype(np.uint8)  # H×W×3
    color[~mask] = 0   # black for invalid pixels
    Image.fromarray(color).save(out_path)


# ─────────────────────────────────────────────
# 3-D  Calibration helpers
# ─────────────────────────────────────────────

def parse_kitti_calib(calib_path: str):
    """
    Parse a KITTI-style calib.txt.
    Returns: K (3×3), T_lid2cam (4×4)
    """
    data = {}
    with open(calib_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line or ":" not in line:
                continue
            key, val = line.split(":", 1)
            data[key.strip()] = np.array([float(x) for x in val.split()])

    # P2 (or P0) → intrinsic K (drop distortion / baseline columns)
    P_key = "P2" if "P2" in data else ("P0" if "P0" in data else None)
    if P_key is None:
        raise KeyError(f"No P2/P0 key in {calib_path}")
    P = data[P_key].reshape(3, 4)
    K = P[:3, :3].copy()

    # Tr_velo_to_cam / Tr_lidar_to_cam / extrinsic_matrix
    tr_key = next(
        (k for k in ["Tr_velo_to_cam", "Tr_lidar_to_cam",
                     "Tr_velo_cam", "extrinsic", "Tr"]
         if k in data), None
    )
    if tr_key is None:
        raise KeyError(f"No lidar→cam transform key in {calib_path}. Keys: {list(data)}")
    Tr = data[tr_key].reshape(3, 4)

    # R0_rect correction (camera 0 rectification) — apply if present
    if "R0_rect" in data or "R_rect" in data:
        rk  = "R0_rect" if "R0_rect" in data else "R_rect"
        R0  = data[rk].reshape(3, 3)
        Tr3 = R0 @ Tr[:3, :3]
        t3  = R0 @ Tr[:3, 3]
        Tr  = np.hstack([Tr3, t3[:, None]])

    T_lid2cam       = np.eye(4, dtype=np.float64)
    T_lid2cam[:3, :] = Tr
    return K.astype(np.float64), T_lid2cam


def parse_yaml_calib(calib_path: str):
    """
    Parse a YAML calibration file. Expects keys:
      camera_matrix (3×3 list-of-lists or flat 9-list)
      lidar_to_camera  (4×4 list-of-lists or flat 16-list)
    Returns: K (3×3), T_lid2cam (4×4)
    """
    with open(calib_path, "r") as f:
        c = yaml.safe_load(f)

    # Support both nested and flat arrays
    def to_np(v, shape):
        a = np.array(v, dtype=np.float64).flatten()
        return a.reshape(shape)

    K_key  = next((k for k in ["camera_matrix", "K", "intrinsic",
                                "camera_intrinsic"] if k in c), None)
    T_key  = next((k for k in ["lidar_to_camera", "T_lidar2cam",
                                "extrinsic", "lidar_to_cam",
                                "lidar2cam"] if k in c), None)
    if K_key is None or T_key is None:
        raise KeyError(f"YAML calib missing required keys. Found: {list(c)}")

    K_raw = c[K_key]
    # Handle OpenCV-style nested dict: {rows:3, cols:3, data:[...]}
    if isinstance(K_raw, dict) and "data" in K_raw:
        K_raw = K_raw["data"]
    K = to_np(K_raw, (3, 3))

    T_raw = c[T_key]
    if isinstance(T_raw, dict) and "data" in T_raw:
        T_raw = T_raw["data"]
    T = to_np(T_raw, (4, 4))

    return K, T


def load_calibration_auto(calib_path: str, manual_K, manual_T):
    """
    Try KITTI, then YAML; fall back to manual arrays.
    Returns: K (3×3 ndarray), T_lid2cam (4×4 ndarray)
    """
    if calib_path and os.path.isfile(calib_path):
        ext = Path(calib_path).suffix.lower()
        try:
            if ext in (".yaml", ".yml", ".json"):
                return parse_yaml_calib(calib_path)
            else:
                return parse_kitti_calib(calib_path)
        except Exception as e:
            warnings.warn(f"Calib load failed ({calib_path}): {e}. Using manual values.")
    else:
        if calib_path:
            warnings.warn(f"Calib not found at '{calib_path}'. Using manual values.")
    K = np.array(manual_K, dtype=np.float64)
    T = np.array(manual_T, dtype=np.float64)
    return K, T


print("✓ Utility functions defined.")

---
## 1 · nuScenes  (.bin  LiDAR)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║              CELL 4 — nuSCENES DEPTH MAP GENERATION                ║
# ╚══════════════════════════════════════════════════════════════════════╝

if not NUSCENES_ENABLED:
    print("nuScenes disabled — skipping.")
else:
    # ── 4-A  Load devkit ─────────────────────────────────────────────
    print(f"Loading nuScenes {NUSCENES_VERSION} from {NUSCENES_DATAROOT} …")
    nusc = NuScenes(version=NUSCENES_VERSION, dataroot=NUSCENES_DATAROOT, verbose=False)
    print(f"   {len(nusc.sample)} samples total")

    # ── 4-B  Select samples by split ────────────────────────────────
    if NUSCENES_SPLIT == "all":
        target_samples = nusc.sample
    else:
        splits = create_splits_scenes()                          # scene-level splits
        split_scenes = set(splits[NUSCENES_SPLIT])
        scene_tokens = {s["token"] for s in nusc.scene
                        if s["name"] in split_scenes}
        target_samples = [s for s in nusc.sample
                          if s["scene_token"] in scene_tokens]

    if NUSCENES_MAX_SAMPLES is not None:
        target_samples = target_samples[:NUSCENES_MAX_SAMPLES]

    print(f"   Processing {len(target_samples)} samples (split='{NUSCENES_SPLIT}')")

    # ── 4-C  Helper: full lidar→camera transform ─────────────────────
    def nuscenes_lidar_to_cam_T(nusc, sd_lidar, sd_cam):
        """
        Compute 4×4 lidar→camera extrinsic, compensating for ego-motion
        between the lidar sweep and camera exposure timestamps.

        Chain:  lidar_frame
                → lidar_ego_frame  (cs_lidar)
                → global_frame     (ep_lidar)    ← ego-motion compensation
                → cam_ego_frame    (ep_cam  ⁻¹)
                → cam_frame        (cs_cam  ⁻¹)
        """
        def _build_T(rotation_wxyz, translation):
            T = np.eye(4, dtype=np.float64)
            T[:3, :3] = Quaternion(rotation_wxyz).rotation_matrix
            T[:3, 3]  = np.array(translation)
            return T

        cs_lidar  = nusc.get("calibrated_sensor", sd_lidar["calibrated_sensor_token"])
        ep_lidar  = nusc.get("ego_pose",          sd_lidar["ego_pose_token"])
        cs_cam    = nusc.get("calibrated_sensor", sd_cam["calibrated_sensor_token"])
        ep_cam    = nusc.get("ego_pose",          sd_cam["ego_pose_token"])

        T_l2e     = _build_T(cs_lidar["rotation"],  cs_lidar["translation"])   # lidar→ego
        T_e2g_l   = _build_T(ep_lidar["rotation"],  ep_lidar["translation"])   # ego→global @lidar
        T_e2g_c   = _build_T(ep_cam["rotation"],    ep_cam["translation"])     # ego→global @cam
        T_c2e     = _build_T(cs_cam["rotation"],    cs_cam["translation"])     # cam_sensor→ego

        if NUSCENES_FULL_TRANSFORM:
            T_lid2cam = (
                np.linalg.inv(T_c2e)
                @ np.linalg.inv(T_e2g_c)
                @ T_e2g_l
                @ T_l2e
            )
        else:
            # Simplified (no ego-motion compensation)
            T_lid2cam = np.linalg.inv(T_c2e) @ T_l2e

        return T_lid2cam

    # ── 4-D  Process every sample ────────────────────────────────────
    out_dir_ns  = os.path.join(OUTPUT_ROOT, "nuscenes", "depth")
    out_dir_col = os.path.join(OUTPUT_ROOT, "nuscenes", "depth_colorized")
    os.makedirs(out_dir_ns,  exist_ok=True)
    os.makedirs(out_dir_col, exist_ok=True)

    errors = []

    for sample in tqdm(target_samples, desc="nuScenes"):
        try:
            # ── Camera sample_data ──────────────────────────────
            sd_cam   = nusc.get("sample_data", sample["data"][NUSCENES_CAMERA])
            img_path = os.path.join(nusc.dataroot, sd_cam["filename"])
            img      = Image.open(img_path)
            img_w, img_h = img.size   # PIL: (W, H)

            # ── LiDAR sample_data ───────────────────────────────
            sd_lidar  = nusc.get("sample_data", sample["data"][NUSCENES_LIDAR])
            lidar_path = os.path.join(nusc.dataroot, sd_lidar["filename"])
            pts_xyz   = load_bin(lidar_path)               # (N,3)

            # ── Camera intrinsic ─────────────────────────────────
            cs_cam    = nusc.get("calibrated_sensor", sd_cam["calibrated_sensor_token"])
            K         = np.array(cs_cam["camera_intrinsic"], dtype=np.float64)  # 3×3

            # ── Extrinsic (full ego-motion-compensated chain) ────
            T_lid2cam = nuscenes_lidar_to_cam_T(nusc, sd_lidar, sd_cam)

            # ── Project ──────────────────────────────────────────
            depth_f = project_lidar_to_depth(
                pts_xyz, K, T_lid2cam, img_h, img_w,
                min_depth=MIN_DEPTH_M, max_depth=MAX_DEPTH_M
            )

            # ── Encode & save ────────────────────────────────────
            depth_u8  = depth_float_to_uint8(depth_f, MAX_DEPTH_M, MIN_DEPTH_M)
            stem      = Path(sd_cam["filename"]).stem   # e.g. n015-...CAM_FRONT__...
            out_path  = os.path.join(out_dir_ns, f"{stem}.png")
            save_depth_uint8(depth_u8, out_path)

            if SAVE_COLORIZED:
                save_depth_colorized(depth_u8,
                    os.path.join(out_dir_col, f"{stem}_color.png"))

            if VERBOSE:
                valid_pct = (depth_u8 > 0).mean() * 100
                print(f"  {stem}: {img_w}×{img_h}  "
                      f"pts={len(pts_xyz):,}  valid={valid_pct:.1f}%")

        except Exception as e:
            msg = f"[ERROR] sample={sample['token']}: {e}"
            errors.append(msg)
            print(msg)

    print(f"\n✓ nuScenes done. Saved to {out_dir_ns}")
    if errors:
        print(f"  {len(errors)} error(s):")
        for e in errors: print(" ", e)

---
## 2 · TIAND  (.pcd  LiDAR)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║               CELL 5 — TIAND DEPTH MAP GENERATION                  ║
# ╚══════════════════════════════════════════════════════════════════════╝
#
# Expected TIAND directory layout:
#
#   <TIAND_ROOT>/
#     <sequence_name>/
#       image_2/          ← front camera images  (*.png / *.jpg)
#       velodyne/         ← LiDAR point clouds   (*.pcd)
#       calib/
#         calib.txt       ← KITTI-style calibration
#
# File pairing is done by matching stem names (frame ID).
# ─────────────────────────────────────────────────────────────────────

if not TIAND_ENABLED:
    print("TIAND disabled — skipping.")
else:
    # ── Discover all sequences ───────────────────────────────────────
    seq_dirs = sorted([
        d for d in Path(TIAND_ROOT).iterdir()
        if d.is_dir()
    ])
    print(f"TIAND: found {len(seq_dirs)} sequence(s) under {TIAND_ROOT}")

    out_dir_td  = os.path.join(OUTPUT_ROOT, "tiand", "depth")
    out_dir_col = os.path.join(OUTPUT_ROOT, "tiand", "depth_colorized")
    os.makedirs(out_dir_td,  exist_ok=True)
    os.makedirs(out_dir_col, exist_ok=True)

    total_saved = 0
    errors      = []

    for seq_dir in seq_dirs:
        seq_name  = seq_dir.name
        img_dir   = seq_dir / TIAND_IMG_SUBDIR
        lidar_dir = seq_dir / TIAND_LIDAR_SUBDIR
        calib_path_seq = seq_dir / TIAND_CALIB_SUBDIR / TIAND_CALIB_FILENAME

        # ── Verify directories ───────────────────────────────────
        if not img_dir.is_dir() or not lidar_dir.is_dir():
            print(f"  [SKIP] {seq_name}: missing image_2 or velodyne dir")
            continue

        # ── Calibration ──────────────────────────────────────────
        # Priority: per-sequence calib.txt > global YAML > manual
        calib_source = str(calib_path_seq) if calib_path_seq.is_file() else TIAND_GLOBAL_CALIB_YAML
        K, T_lid2cam = load_calibration_auto(
            calib_source, TIAND_MANUAL_K, TIAND_MANUAL_T_LID2CAM
        )

        # ── Match image ↔ LiDAR frames by stem ──────────────────
        img_files   = {p.stem: p for p in sorted(img_dir.glob("*"))
                       if p.suffix.lower() in (".png", ".jpg", ".jpeg")}
        lidar_files = {p.stem: p for p in sorted(lidar_dir.glob("*.pcd"))}
        paired_stems = sorted(set(img_files) & set(lidar_files))

        if not paired_stems:
            print(f"  [SKIP] {seq_name}: no paired img+lidar frames found")
            continue

        if TIAND_MAX_SAMPLES is not None:
            paired_stems = paired_stems[:TIAND_MAX_SAMPLES]

        seq_out_d = os.path.join(out_dir_td,  seq_name)
        seq_out_c = os.path.join(out_dir_col, seq_name)
        os.makedirs(seq_out_d, exist_ok=True)
        os.makedirs(seq_out_c, exist_ok=True)

        for stem in tqdm(paired_stems, desc=f"TIAND/{seq_name}", leave=False):
            try:
                img      = Image.open(str(img_files[stem]))
                img_w, img_h = img.size

                pts_xyz  = load_pcd(str(lidar_files[stem]))

                depth_f  = project_lidar_to_depth(
                    pts_xyz, K, T_lid2cam, img_h, img_w,
                    min_depth=MIN_DEPTH_M, max_depth=MAX_DEPTH_M
                )
                depth_u8 = depth_float_to_uint8(depth_f, MAX_DEPTH_M, MIN_DEPTH_M)

                save_depth_uint8(depth_u8, os.path.join(seq_out_d, f"{stem}.png"))
                if SAVE_COLORIZED:
                    save_depth_colorized(depth_u8,
                        os.path.join(seq_out_c, f"{stem}_color.png"))

                if VERBOSE:
                    valid_pct = (depth_u8 > 0).mean() * 100
                    print(f"  {seq_name}/{stem}: {img_w}×{img_h}  "
                          f"pts={len(pts_xyz):,}  valid={valid_pct:.1f}%")

                total_saved += 1

            except Exception as e:
                msg = f"[ERROR] {seq_name}/{stem}: {e}"
                errors.append(msg)
                print(msg)

    print(f"\n✓ TIAND done. {total_saved} depth maps saved to {out_dir_td}")
    if errors:
        print(f"  {len(errors)} error(s):")
        for e in errors: print(" ", e)

---
## 3 · IDD  (.npy  LiDAR)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║                IDD DEPTH MAP GENERATION                            ║
# ╚══════════════════════════════════════════════════════════════════════╝
#
# Expected IDD layout:
#
#   <IDD_ROOT>/
#     <split>/                    e.g. train/, val/
#       <segment_id>/
#         leftImg8bit/            ← front camera images  (*.png)
#         lidar/                  ← point clouds          (*.npy)
#     calib/
#       idd_calib.yaml            ← single global calibration
#
# .npy arrays are expected as (N,3) or (N,4+) float32 [x,y,z,(intensity)].
# ─────────────────────────────────────────────────────────────────────

if not IDD_ENABLED:
    print("IDD disabled — skipping.")
else:
    # ── Load single global calibration ────────────────────────────
    K_idd, T_lid2cam_idd = load_calibration_auto(
        IDD_CALIB_FILE, IDD_MANUAL_K, IDD_MANUAL_T_LID2CAM
    )
    print(f"IDD calibration loaded.")
    print(f"  K =\n{K_idd}")
    print(f"  T_lid2cam =\n{T_lid2cam_idd}")

    out_dir_idd = os.path.join(OUTPUT_ROOT, "idd", "depth")
    out_dir_col = os.path.join(OUTPUT_ROOT, "idd", "depth_colorized")
    os.makedirs(out_dir_idd, exist_ok=True)
    os.makedirs(out_dir_col, exist_ok=True)

    total_saved = 0
    errors      = []

    for split in IDD_SPLITS:
        split_dir = Path(IDD_ROOT) / split
        if not split_dir.is_dir():
            print(f"  [SKIP] split '{split}' not found at {split_dir}")
            continue

        # Each sub-directory is a segment / drive sequence
        seg_dirs = sorted([d for d in split_dir.iterdir() if d.is_dir()])
        print(f"IDD split '{split}': {len(seg_dirs)} segment(s)")

        for seg_dir in seg_dirs:
            seg_name  = seg_dir.name
            img_dir   = seg_dir / IDD_IMG_SUBDIR
            lidar_dir = seg_dir / IDD_LIDAR_SUBDIR

            if not img_dir.is_dir() or not lidar_dir.is_dir():
                if VERBOSE:
                    print(f"  [SKIP] {split}/{seg_name}: missing image or lidar dir")
                continue

            # ── Match by stem ────────────────────────────────────
            img_files   = {p.stem: p for p in sorted(img_dir.glob("*.png"))}
            lidar_files = {p.stem: p for p in sorted(lidar_dir.glob("*.npy"))}

            # IDD img stems often end in '_leftImg8bit'; strip that suffix
            def normalise_stem(s: str) -> str:
                return s.replace("_leftImg8bit", "").replace("_lidar", "")

            img_norm   = {normalise_stem(k): v for k, v in img_files.items()}
            lidar_norm = {normalise_stem(k): v for k, v in lidar_files.items()}
            paired     = sorted(set(img_norm) & set(lidar_norm))

            if not paired:
                if VERBOSE:
                    print(f"  [SKIP] {split}/{seg_name}: no paired frames")
                continue

            if IDD_MAX_SAMPLES is not None:
                paired = paired[:IDD_MAX_SAMPLES]

            seg_out_d = os.path.join(out_dir_idd, split, seg_name)
            seg_out_c = os.path.join(out_dir_col, split, seg_name)
            os.makedirs(seg_out_d, exist_ok=True)
            os.makedirs(seg_out_c, exist_ok=True)

            for ns in tqdm(paired, desc=f"IDD/{split}/{seg_name}", leave=False):
                try:
                    img      = Image.open(str(img_norm[ns]))
                    img_w, img_h = img.size

                    pts_xyz  = load_npy(str(lidar_norm[ns]))

                    depth_f  = project_lidar_to_depth(
                        pts_xyz, K_idd, T_lid2cam_idd, img_h, img_w,
                        min_depth=MIN_DEPTH_M, max_depth=MAX_DEPTH_M
                    )
                    depth_u8 = depth_float_to_uint8(depth_f, MAX_DEPTH_M, MIN_DEPTH_M)

                    save_depth_uint8(depth_u8,
                        os.path.join(seg_out_d, f"{ns}.png"))
                    if SAVE_COLORIZED:
                        save_depth_colorized(depth_u8,
                            os.path.join(seg_out_c, f"{ns}_color.png"))

                    if VERBOSE:
                        valid_pct = (depth_u8 > 0).mean() * 100
                        print(f"  {split}/{seg_name}/{ns}: {img_w}×{img_h}  "
                              f"pts={len(pts_xyz):,}  valid={valid_pct:.1f}%")

                    total_saved += 1

                except Exception as e:
                    msg = f"[ERROR] {split}/{seg_name}/{ns}: {e}"
                    errors.append(msg)
                    print(msg)

    print(f"\n✓ IDD done. {total_saved} depth maps saved to {out_dir_idd}")
    if errors:
        print(f"  {len(errors)} error(s):")
        for e in errors: print(" ", e)

---
## QC — Visual Verification

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║                  CELL 7 — QC VISUALISATION                         ║
# ╚══════════════════════════════════════════════════════════════════════╝
#
# For each dataset, finds one saved depth map and plots:
#   (a) raw uint8 depth PNG
#   (b) colourised version
#   (c) depth histogram (valid pixels only)
#   (d) decoded metric depth statistics
# ─────────────────────────────────────────────────────────────────────

def qc_plot(dataset_label: str, output_subdir: str, max_depth: float = MAX_DEPTH_M):
    """Find the first depth PNG in output_subdir and plot QC figures."""
    hits = list(Path(output_subdir).rglob("*.png"))
    # Exclude colorised versions
    hits = [p for p in hits if "_color" not in p.stem and "colorized" not in str(p)]
    if not hits:
        print(f"[{dataset_label}] No depth PNGs found in {output_subdir}")
        return

    sample_path = hits[0]
    d_u8 = np.array(Image.open(str(sample_path)), dtype=np.uint8)  # H×W
    valid = d_u8 > 0

    # Decode to metres for statistics
    d_m = d_u8.astype(np.float32) / 255.0 * max_depth

    print(f"\n{'─'*60}")
    print(f" {dataset_label}  →  {sample_path.name}")
    print(f"{'─'*60}")
    print(f"  Image shape      : {d_u8.shape[1]}×{d_u8.shape[0]} (W×H)")
    print(f"  Valid pixels     : {valid.sum():,} / {d_u8.size:,} "
          f"({valid.mean()*100:.2f}%)")
    if valid.any():
        vals = d_m[valid]
        print(f"  Depth range [m]  : {vals.min():.2f} – {vals.max():.2f}")
        print(f"  Depth mean  [m]  : {vals.mean():.2f}")
        print(f"  Depth median[m]  : {np.median(vals):.2f}")

    # ── Figure ──────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    fig.suptitle(f"{dataset_label}  —  {sample_path.name}", fontsize=12)

    # (a) raw uint8
    axes[0].imshow(d_u8, cmap="gray", vmin=0, vmax=255)
    axes[0].set_title("uint8 depth (grey)")
    axes[0].axis("off")

    # (b) jet-colourised
    jet_img = np.zeros((*d_u8.shape, 3), dtype=np.uint8)
    if valid.any():
        jet_img[valid] = (cm.jet(d_u8[valid].astype(np.float32) / 255.0)[:, :3] * 255).astype(np.uint8)
    axes[1].imshow(jet_img)
    axes[1].set_title("Jet colourised depth")
    axes[1].axis("off")

    # (c) depth histogram
    if valid.any():
        axes[2].hist(d_m[valid], bins=80, color="steelblue", edgecolor="none")
        axes[2].set_xlabel("Depth [m]")
        axes[2].set_ylabel("LiDAR returns")
        axes[2].set_title("Metric depth histogram")
        axes[2].axvline(np.median(d_m[valid]), color="red", lw=1.5,
                        label=f"median={np.median(d_m[valid]):.1f} m")
        axes[2].legend(fontsize=9)
    else:
        axes[2].text(0.5, 0.5, "No valid returns", ha="center", va="center")

    plt.tight_layout()
    plt.show()


# ── Run QC for each enabled dataset ─────────────────────────────────
if NUSCENES_ENABLED:
    qc_plot("nuScenes", os.path.join(OUTPUT_ROOT, "nuscenes", "depth"))

if TIAND_ENABLED:
    qc_plot("TIAND",    os.path.join(OUTPUT_ROOT, "tiand",    "depth"))

if IDD_ENABLED:
    qc_plot("IDD",      os.path.join(OUTPUT_ROOT, "idd",      "depth"))

---
## Decode Reference
```python
# To decode a saved uint8 depth PNG back to metric metres:
import numpy as np
from PIL import Image

MAX_DEPTH_M = 80.0          # must match encoding config

d_u8   = np.array(Image.open("frame.png"), dtype=np.uint8)   # H×W
valid  = d_u8 > 0
d_m    = np.zeros_like(d_u8, dtype=np.float32)
d_m[valid] = d_u8[valid].astype(np.float32) / 255.0 * MAX_DEPTH_M
# d_m: float32 depth map in metres; 0.0 = invalid
```